In [ ]:
# kernel: baseline_model 
# Python: 3.11.14
# dtw: 1.4.0 (deprecated package)

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from dtw import dtw
import copy

# from keras.models import Sequential
import matplotlib.pyplot as plt
%matplotlib inline

# import tensorflow as tf
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed, Dropout

import plotly.express as px

In [ ]:
machines = ["SL40309_015"]
programs = ["O0005(5303-005-C)"]
crs = ["CR1_to_CR4", "CR5_to_CR8", "CR9_to_CR12", "CR12_to_CR16"]
base_name = "data"
paths = []

# data_SL40309_015_O0005(5303-005-C)_CR12_to_CR16

for machine in machines:
    for program in programs:
        for cr in crs:
            path = f"{base_name}_{machine}_{program}_{cr}.csv"
            paths.append(path)
print(paths)

dfs = []

for path in paths:
    data = pd.read_csv(f"D:/baseline_improvement-main/data/{path}")
    dfs.append(data)
    # print(len(data))

data = pd.concat(dfs, ignore_index=True)

data = data.pivot(
    index=["custom_id", "timestamp", "status", "cr"],
    columns=["name", "workstationcomponent"],
    values="value",
)
data = data.reset_index()
data.columns = [f"{col}_{comp}" if comp != "" else col for col, comp in data.columns]
data.columns

['data_SL40309_015_O0005(5303-005-C)_CR1_to_CR4.csv', 'data_SL40309_015_O0005(5303-005-C)_CR5_to_CR8.csv', 'data_SL40309_015_O0005(5303-005-C)_CR9_to_CR12.csv', 'data_SL40309_015_O0005(5303-005-C)_CR12_to_CR16.csv']


Index(['custom_id', 'timestamp', 'status', 'cr', 'gcode.ncode_Path_Path_1',
       'position_Linear_Z', 'toolnumber_Path_Path_1', 'position_Linear_X',
       'gcode.toolSlotId_Path_Path_1', 'programcomment_Path_Path_1',
       'pathfeedrate_Path_Path_1', 'gcode.program_Path_Path_1',
       'gcode.mcode_Path_Path_1', 'execution_Path_Path_1',
       'spindlespeed_actual_Rotary_C5', 'load_Rotary_C5',
       'spindlerotating_Rotary_C5', 'load_Linear_X',
       'gcode.gearselect_Path_Path_1', 'counter.last30seventscount_nan',
       'gcode.coolant_Path_Path_1', 'coolant_Coolant_Coolant', 'load_Linear_Z',
       'rapidoverride_Path_Path_1', 'controllermode_Path_Path_1',
       'operationmode_Path_Path_1', 'cutting_Path_Path_1', 'heartbeat_datatap',
       'line_Path_Path_1', 'partcount_Path_Path_1',
       'gcode.programname_Path_Path_1', 'fault_Path_Path_1',
       'gcode.spindle_Path_Path_1', 'jogoverride_Path_Path_1'],
      dtype='str')

In [ ]:
split_columns = data["status"].str.split("/", expand=True)
data[["program_name", "nsequence", "execution"]] = split_columns[[0, 1, 2]]

data["load_Rotary_C5"] = data["load_Rotary_C5"].astype(float)
data["position_Linear_Z"] = data["position_Linear_Z"].astype(float)
data["position_Linear_X"] = data["position_Linear_X"].astype(float)
data["spindlespeed_actual_Rotary_C5"] = data["spindlespeed_actual_Rotary_C5"].astype(
    float
)
data["pathfeedrate_Path_Path_1"] = data["pathfeedrate_Path_Path_1"].astype(float)

data = data.ffill()

In [ ]:
# datapoints where machining == True
data = data[data["timestamp"] < "2024-04-12"]
data = data[
    (data["execution"] == "ACTIVE")
    & (data["spindlespeed_actual_Rotary_C5"] != 0)
    & (data["pathfeedrate_Path_Path_1"] != 0)
    & (data["load_Rotary_C5"] > 0)
    & (data["pathfeedrate_Path_Path_1"] <= 1000)
]

In [30]:
len(data)

85606

In [31]:
data.tail(5)

,custom_id,timestamp,status,cr,gcode.ncode_Path_Path_1,position_Linear_Z,toolnumber_Path_Path_1,position_Linear_X,gcode.toolSlotId_Path_Path_1,programcomment_Path_Path_1,...,heartbeat_datatap,line_Path_Path_1,partcount_Path_Path_1,gcode.programname_Path_Path_1,fault_Path_Path_1,gcode.spindle_Path_Path_1,jogoverride_Path_Path_1,program_name,nsequence,execution
119737,20097830,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,14.1226,1212,19.1076,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE
119738,20097831,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,14.1226,1212,-0.0885,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE
119739,20097832,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,14.1226,1212,-0.0885,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE
119740,20097833,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,-5.6411,1212,-0.0885,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE
119741,20097834,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,-5.6411,1212,-0.0885,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE


In [ ]:
dfN10_load = data[
    [
        "timestamp",
        "load_Rotary_C5",
        "position_Linear_X",
        "position_Linear_Z",
        "spindlespeed_actual_Rotary_C5",
        "pathfeedrate_Path_Path_1",
        "execution",
        "program_name",
        "nsequence",
        "cr",
    ]
]

In [33]:
len(dfN10_load)

85606

In [ ]:
dfN10_load = dfN10_load.drop_duplicates(subset=["timestamp"], keep="first")
dfN10_load.reset_index(drop=True)
len(dfN10_load)

9937

In [ ]:
dfN10_load['index'] = dfN10_load.groupby(['cr']).cumcount()

In [ ]:
fig = px.line(dfN10_load, x="index", y="load_Rotary_C5", color="cr")
fig.update_layout(title_text="Load Vs Index")
fig.show()

In [ ]:
LE = LabelEncoder()

dfN10_load['execution'] = LE.fit_transform(dfN10_load['execution'])
dfN10_load.fillna(0, inplace=True)

,timestamp,load_Rotary_C5,position_Linear_X,position_Linear_Z,spindlespeed_actual_Rotary_C5,pathfeedrate_Path_Path_1,execution,program_name,nsequence,cr,index
46,2024-03-11 14:01:19.408233,2.0,18.8794,12.5102,35.0,238.02,0,O0005(5303-005-C),N10,CR-1,0
49,2024-03-11 14:01:29.432814,2.0,18.8794,12.5102,31.0,238.02,0,O0005(5303-005-C),N10,CR-1,1
63,2024-03-11 14:01:37.451878,1.0,16.9492,4.7948,34.0,238.02,0,O0005(5303-005-C),N10,CR-1,2
64,2024-03-11 14:01:44.468672,1.0,16.9492,4.7948,33.0,238.02,0,O0005(5303-005-C),N10,CR-1,3
78,2024-03-11 14:01:52.489886,1.0,16.5496,3.1977,34.0,14.81,0,O0005(5303-005-C),N10,CR-1,4
...,...,...,...,...,...,...,...,...,...,...,...
119559,2024-03-12 06:39:29.791527,1.0,16.2506,-0.1245,35.0,0.49,0,O0005(5303-005-C),N40,CR-16,524
119565,2024-03-12 06:39:35.803604,1.0,16.2758,-0.1586,29.0,1.39,0,O0005(5303-005-C),N40,CR-16,525
119575,2024-03-12 06:39:45.824041,1.0,17.9761,1.7775,29.0,237.16,0,O0005(5303-005-C),N40,CR-16,526
119579,2024-03-12 06:39:49.830634,1.0,19.1076,7.9186,29.0,237.16,0,O0005(5303-005-C),N40,CR-16,527


In [ ]:
scaler1 = MinMaxScaler()

dfN10_load[['load_Rotary_C5_temp']]= scaler1.fit_transform(dfN10_load[['load_Rotary_C5']])

In [39]:
dfN10_load.head()

,timestamp,load_Rotary_C5,position_Linear_X,position_Linear_Z,spindlespeed_actual_Rotary_C5,pathfeedrate_Path_Path_1,execution,program_name,nsequence,cr,index,load_Rotary_C5_temp
46,2024-03-11 14:01:19.408233,2.0,18.8794,12.5102,35.0,238.02,0,O0005(5303-005-C),N10,CR-1,0,0.014493
49,2024-03-11 14:01:29.432814,2.0,18.8794,12.5102,31.0,238.02,0,O0005(5303-005-C),N10,CR-1,1,0.014493
63,2024-03-11 14:01:37.451878,1.0,16.9492,4.7948,34.0,238.02,0,O0005(5303-005-C),N10,CR-1,2,0.000000
64,2024-03-11 14:01:44.468672,1.0,16.9492,4.7948,33.0,238.02,0,O0005(5303-005-C),N10,CR-1,3,0.000000
78,2024-03-11 14:01:52.489886,1.0,16.5496,3.1977,34.0,14.81,0,O0005(5303-005-C),N10,CR-1,4,0.000000


In [ ]:
dfN10_load['cr'].value_counts()

cr
CR-1     783
CR-10    778
CR-13    744
CR-4     740
CR-2     665
CR-3     664
CR-11    655
CR-12    654
CR-8     538
CR-14    537
CR-5     535
CR-6     531
CR-7     531
CR-9     530
CR-16    529
CR-15    523
Name: count, dtype: int64

In [ ]:
dfN10_load['nsequence'].value_counts()

nsequence
N20    3729
N30    3205
N10    2477
N40     526
Name: count, dtype: int64

In [42]:
dfN10_load.columns

Index(['timestamp', 'load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z',
       'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1',
       'execution', 'program_name', 'nsequence', 'cr', 'index',
       'load_Rotary_C5_temp'],
      dtype='str')

In [ ]:
# dfN10_load.head()
# Data required for DTW eval :

req_data = dfN10_load[
    [
        "timestamp",
        "load_Rotary_C5_temp",
        "execution",
        "program_name",
        "nsequence",
        "cr",
        "index",
    ]
]
req_data.head()

,timestamp,load_Rotary_C5_temp,execution,program_name,nsequence,cr,index
46,2024-03-11 14:01:19.408233,0.014493,0,O0005(5303-005-C),N10,CR-1,0
49,2024-03-11 14:01:29.432814,0.014493,0,O0005(5303-005-C),N10,CR-1,1
63,2024-03-11 14:01:37.451878,0.000000,0,O0005(5303-005-C),N10,CR-1,2
64,2024-03-11 14:01:44.468672,0.000000,0,O0005(5303-005-C),N10,CR-1,3
78,2024-03-11 14:01:52.489886,0.000000,0,O0005(5303-005-C),N10,CR-1,4


In [44]:
path = "D:/baseline_improvement-main/dtw_improvement/data/preprocessed_data.csv"
req_data.to_csv(path)